In [15]:
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

print(f"PyTorch version loaded: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computer brain using: {DEVICE}")

"""
================================================================================
NeuralProt — Surgical Folder Rescuer
================================================================================
What this code does:
1. It looks inside your processed data folder for the nested 'serine' folders.
2. It safely renames and pulls them out to the main directory.
3. It makes them match your flat 'serine_threonine...' format perfectly.
"""

import os
import shutil

# ── WORKSPACE DISK PATHS ──────────────────────────────────────────────────────
PROCESSED_DIR   = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/processed_data"
OUTPUT_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/models"
ASSIGNMENT_PATH = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/go_group_assignment_v3.json"

# ── STUDY PACING SETTINGS ────────────────────────────────────────────────────
BATCH_SIZE      = 32
EPOCHS          = 100
LR              = 0.001
WEIGHT_DECAY    = 1e-4
POS_WEIGHT_CLIP = 10000

# Automatically load all 370+ active group folders from your new map file
with open(ASSIGNMENT_PATH, "r") as f:
    _blueprint_data = json.load(f)
TRAIN_GROUPS = list(_blueprint_data["groups"].keys())

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Notebook memory initialized! Loaded {len(TRAIN_GROUPS)} training groups safely.")

PyTorch version loaded: 2.9.1+cpu
Computer brain using: cpu
✓ Notebook memory initialized! Loaded 375 training groups safely.


In [16]:
"""
================================================================================
NeuralProt — Pre-Training Blueprint Inspector (Cell 1)
================================================================================
What this cell does:
1. It opens your master group assignment map to find all 373 folders.
2. It checks the actual data files inside every folder without loading the 
   whole file into memory (using a speed trick called 'mmap_mode').
3. It prints a neat summary table and alerts you if any group is broken.
"""

# ── 1. THE INSPECTOR ENGINE ───────────────────────────────────────────────────


def inspect_all_groups():
    # Verify the blueprint map exists
    if not os.path.exists(ASSIGNMENT_PATH):
        print(f"❌ ERROR: Cannot find blueprint file at {ASSIGNMENT_PATH}")
        return

    with open(ASSIGNMENT_PATH, "r") as f:
        data = json.load(f)
    group_list = list(data["groups"].keys())

    print("=" * 75)
    print(f"NeuralProt Inspector — Scanning {len(group_list)} Folders")
    print("=" * 75)
    print(f"  {'Group Name':<45} | {'Proteins':<8} | {'GO Terms':<8} | {'Features':<8}")
    print("-" * 75)

    broken_groups = []
    total_proteins_counted = 0

    # Loop through all 373 groups one by one
    for group_name in group_list:
        safe_name = group_name.replace(":", "_")
        group_folder = os.path.join(PROCESSED_DIR, safe_name)

        feat_file = os.path.join(group_folder, "features.npy")
        label_file = os.path.join(group_folder, "labels.npz")

        # Check if the physical files actually exist on the disk
        if not os.path.exists(feat_file) or not os.path.exists(label_file):
            broken_groups.append((group_name, "Missing data files"))
            continue

        try:
            # Look inside features.npy using a fast 'read-only' look trick
            feat_matrix = np.load(feat_file, mmap_mode="r")
            n_proteins = feat_matrix.shape[0]
            n_features = feat_matrix.shape[1]

            # Look inside labels.npz
            with np.load(label_file) as archive:
                label_matrix = archive["labels"]
                n_go_terms = label_matrix.shape[1]
                n_label_proteins = label_matrix.shape[0]

            # Safety Check: Make sure feature rows match label rows perfectly
            if n_proteins != n_label_proteins:
                broken_groups.append((group_name, "Row mismatch error"))
                continue

            total_proteins_counted += n_proteins

            # Print the single clean row line for this group
            # (Truncates names longer than 42 letters so the table stays neat)
            short_name = (
                safe_name if len(safe_name) <= 42 else safe_name[:39] + "..."
            )
            print(
                f"  {short_name:<45} | {n_proteins:<8} | {n_go_terms:<8} | {n_features:<8}"
            )

        except Exception as e:
            broken_groups.append((group_name, f"File read crash: {str(e)}"))

    # ── 2. THE FINAL ACCOUNTING WARNING ALERTS ────────────────────────────────
    print("=" * 75)
    print(" Scan Complete Summary:")
    print(f"   -> Total active folders checked: {len(group_list)}")
    print(f"   -> Sum of all protein rows across groups: {total_proteins_counted:,}")
    print("=" * 75)

    if broken_groups:
        print("\n🚨 RED ALERT! Found broken groups that will cause training crashes:")
        for name, issue in broken_groups:
            print(f"   ⚠️ Group: {name} -> Reason: {issue}")
        print("\nAction Required: Do not start training until you fix these folders!")
    else:
        print(
            "\n✅ PERFECT SCORE: All 373 groups match their shapes perfectly. It is 100% safe to split your data and begin training!"
        )


# Run the inspector
inspect_all_groups()

NeuralProt Inspector — Scanning 375 Folders
  Group Name                                    | Proteins | GO Terms | Features
---------------------------------------------------------------------------
  positive_regulation_of_DNA-templated_tr...    | 4796     | 16       | 498     
  regulation_of_mRNA_stability_GO_0043488       | 466      | 16       | 498     
  heterochromatin_formation_GO_0031507          | 609      | 16       | 498     
  pattern_recognition_receptor_signaling_...    | 306      | 16       | 498     
  regulation_of_protein_kinase_activity_G...    | 429      | 18       | 498     
  positive_regulation_of_lymphocyte_diffe...    | 278      | 15       | 498     
  ribonucleoside_monophosphate_biosynthet...    | 483      | 16       | 498     
  fatty_acid_biosynthetic_process_GO_0006633    | 1152     | 15       | 498     
  tRNA_aminoacylation_for_protein_transla...    | 275      | 15       | 498     
  regulation_of_translation_GO_0006417          | 1539     | 26       

In [17]:
"""
================================================================================
NeuralProt — PyTorch Data Loader & Splitting System (Upgraded to 498 Features)
================================================================================
What this script does:
1. It opens your safe Windows group folders and reads the clean data matrices.
2. It unzips the 'labels.npz' files to see which proteins do which jobs.
3. It chops your protein lists into three distinct piles: 80% for training the 
   AI, 10% for testing it during class, and 10% locked away for final grading.
4. It feeds these groups into PyTorch as numbers, requiring zero text tokenizers.
"""

class ProteinDataset(Dataset):
    """Holds our precalculated 498-dimensional physical numbers for PyTorch."""
    def __init__(self, indices, feature_matrix, label_matrix):
        self.indices        = indices
        self.feature_matrix = feature_matrix
        self.label_matrix   = label_matrix
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, idx):
        i = self.indices[idx]
        features = torch.tensor(self.feature_matrix[i], dtype=torch.float32)
        label    = torch.tensor(self.label_matrix[i],   dtype=torch.float32)
        return features, label
 
def load_group_data(group_name, processed_dir):
    """Safely opens your Windows folders to load numbers without crashing."""
    # FIXED: Clean out both colons AND forward slashes so Windows paths work!
    safe_group_name = group_name.replace(":", "_").replace("/", "_")
    group_dir = os.path.join(processed_dir, safe_group_name)
 
    required = ["accessions.json", "labels.npz", "terms.json", "features.npy", "metadata.json"]
    for fname in required:
        if not os.path.exists(os.path.join(group_dir, fname)):
            raise FileNotFoundError(f"Missing file: {fname} inside {group_dir}")
 
    with open(os.path.join(group_dir, "accessions.json")) as f:
        accessions = json.load(f)
 
    with np.load(os.path.join(group_dir, "labels.npz")) as archive:
        label_matrix = archive["labels"]
        
    feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
    with open(os.path.join(group_dir, "terms.json")) as f:
        go_term_list = json.load(f)
    with open(os.path.join(group_dir, "metadata.json")) as f:
        metadata = json.load(f)
 
    return accessions, feature_matrix, label_matrix, go_term_list, metadata
 
def build_dataloaders(feature_matrix, label_matrix, batch_size, group_dir):
    """Slices data into study piles and validation quiz piles safely."""
    n       = len(label_matrix)
    indices = list(range(n))

    train_val_idx, test_idx = train_test_split(indices, test_size=0.1, random_state=42)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.111, random_state=42)

    test_idx_path = os.path.join(group_dir, "test_idx.json")
    if not os.path.exists(test_idx_path):
        with open(test_idx_path, "w") as f:
            json.dump(test_idx, f)

    train_ds = ProteinDataset(train_idx, feature_matrix, label_matrix)
    val_ds   = ProteinDataset(val_idx,   feature_matrix, label_matrix)

    # FIXED: Added drop_last=True to prevent the single-protein batch norm crash!
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

    return train_loader, val_loader, len(train_ds), len(val_ds)
 
def compute_pos_weights(label_matrix, clip_max=10000.0):
    n          = label_matrix.shape[0]
    pos_counts = label_matrix.sum(axis=0) + 1e-6
    neg_counts = n - pos_counts
    weights    = np.clip(neg_counts / pos_counts, 1.0, clip_max)
    return torch.tensor(weights, dtype=torch.float32)
 
print("✓ Dataset loader tools ready.")

✓ Dataset loader tools ready.


In [18]:
"""
================================================================================
NeuralProt — Upgraded 498-Feature Neural Network (ProteinMLP)
================================================================================
What this code does:
1. It builds a 4-layer feedforward neural network (a stacked line of math boxes).
2. It takes your 498 physical shape numbers as input.
3. It uses 'ReLU' filters to help the model learn complex curved shapes.
4. It uses 'Dropout' safety drops to stop the model from memorizing the answers.
5. It outputs raw guess numbers (logits) for each job tag inside the group box.
"""

class ProteinMLP(nn.Module):
    """4-layer neural network optimized for 498 protein shape inputs."""
    def __init__(self, input_dim=498, num_classes=128, dropout=0.3):
        super().__init__()
 
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
 
            nn.Linear(256, num_classes),
        )
 
    def forward(self, x):
        return self.network(x)
 
print("✓ ProteinMLP brain layout defined with 498 inputs.")

✓ ProteinMLP brain layout defined with 498 inputs.


In [19]:
"""
================================================================================
NeuralProt — Master Network Trainer & Exam Grader (With Auto-Resume)
================================================================================
What this code does:
1. 'evaluate': Acts like a practice quiz. It tests the network on unseen
   proteins and scores it using an F1-score (how well it balances correct guesses).
2. 'train_group': The master studying loop. It lets the network look at data,
   makes adjustments when it makes mistakes, and saves its progress.
3. It saves a '_best.pt' file when the model gets a new high score, and a 
   '_resume.pt' file after every single round so you never lose your work.
"""

def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_targets = []
 
    with torch.no_grad():
        for seqs, labels in loader:
            seqs   = seqs.to(device)
            labels = labels.to(device)
            logits = model(seqs)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).float()
            all_preds.append(preds.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
 
    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
 
    f1        = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    precision = precision_score(all_targets, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_targets, all_preds, average="macro", zero_division=0)
    avg_loss  = total_loss / len(loader)
 
    return avg_loss, f1, precision, recall
 
def train_group(group_name, train_loader, val_loader, num_classes,
                pos_weights, device, output_dir, epochs, lr, weight_decay):
    
    safe_group_name = group_name.replace(":", "_").replace("/", "_")
    best_path   = os.path.join(output_dir, f"{safe_group_name}_best.pt")
    resume_path = os.path.join(output_dir, f"{safe_group_name}_resume.pt")
    log_path    = os.path.join(output_dir, f"{safe_group_name}_log.json")
 
    model     = ProteinMLP(num_classes=num_classes).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)
 
    start_epoch = 1
    best_val_f1 = -1.0
    log         = []
 
    if os.path.exists(resume_path):
        checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
        if checkpoint.get("training_complete", False):
            print(f"  ✓ Already completed. Skipping: {group_name} (F1: {checkpoint['best_val_f1']:.4f})")
            return checkpoint["best_val_f1"]
 
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        start_epoch = checkpoint["epoch"] + 1
        best_val_f1 = checkpoint["best_val_f1"]
        log         = checkpoint.get("log", [])
 
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        epoch_loss = 0.0
        t_start    = time.time()
 
        for seqs, labels in train_loader:
            seqs   = seqs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            logits = model(seqs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
 
        avg_train_loss = epoch_loss / len(train_loader)
        val_loss, val_f1, val_p, val_r = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t_start
 
        improved = ""
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "val_f1": val_f1, "val_loss": val_loss,
                "num_classes": num_classes, "group_name": group_name,
            }, best_path)
            improved = "  ← best"

            # ── COPY terms.json INTO MODELS FOLDER SO BACKEND CAN FIND IT ──
            import shutil
            src_terms  = os.path.join(PROCESSED_DIR, safe_group_name, "terms.json")
            dest_terms = os.path.join(output_dir, f"{safe_group_name}_terms.json")
            if os.path.exists(src_terms):
                shutil.copy2(src_terms, dest_terms)
 
        print(f"  Round {epoch:>2}/{epochs} | Loss: {avg_train_loss:.4f} | Quiz Loss: {val_loss:.4f} | F1: {val_f1:.4f} {improved}")
 
        log.append({"epoch": epoch, "train_loss": round(avg_train_loss, 6), "val_loss": round(val_loss, 6), "val_f1": round(val_f1, 6)})
        with open(log_path, "w") as f: json.dump(log, f, indent=2)
 
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(), "best_val_f1": best_val_f1,
            "num_classes": num_classes, "group_name": group_name, "log": log, "training_complete": (epoch == epochs),
        }, resume_path)
 
    return best_val_f1

print("✓ Training loops hardened and ready.")

✓ Training loops hardened and ready.


In [20]:
"""
================================================================================
NeuralProt — Master Training Orchestrator Loop (Final Cell)
================================================================================
What this code does:
1. It loops through all 373 of your packed model boxes one by one.
2. It opens the data matrices and sets up the balanced study multipliers.
3. It splits the proteins into study piles and practice quiz piles.
4. It calls the training brain layers and saves a report card summary.
5. If a single box fails, it catches the error and moves on to the next box 
   instead of crashing your whole multi-hour run.
"""

# This dictionary acts as our final master report card folder box
summary = {}
 
for group_name in TRAIN_GROUPS:
    try:
        safe_group_name = group_name.replace(":", "_").replace("/", "_")
 
        # 1. Load your processed numbers from disk
        accessions, feature_matrix, label_matrix, go_term_list, metadata = \
            load_group_data(group_name, PROCESSED_DIR)
 
        print(f"\nLoaded folder: {safe_group_name}")
        print(f"  Proteins counted : {len(accessions):,}")
 
        # 2. Calculate balance multipliers
        pos_weights = compute_pos_weights(label_matrix, clip_max=POS_WEIGHT_CLIP)
 
        # 3. Cut lists into study piles safely
        group_dir = os.path.join(PROCESSED_DIR, safe_group_name)
        train_loader, val_loader, n_train, n_val = build_dataloaders(
                feature_matrix, label_matrix, BATCH_SIZE, group_dir,
        )
        n_test = len(label_matrix) - n_train - n_val
        print(f"  Slices: Train={n_train:,} | Val={n_val:,} | Held-out Test={n_test:,}")
 
        # 4. Fire up the training loops
        best_f1 = train_group(
            group_name   = group_name,
            train_loader = train_loader,
            val_loader   = val_loader,
            num_classes  = len(go_term_list),
            pos_weights  = pos_weights,
            device       = DEVICE,
            output_dir   = OUTPUT_DIR,
            epochs       = EPOCHS,
            lr           = LR,
            weight_decay = WEIGHT_DECAY,
        )
 
        summary[group_name] = {
            "n_proteins":  len(accessions),
            "n_go_terms":  len(go_term_list),
            "best_val_f1": round(best_f1, 4),
            "status":      "completed",
        }
 
    except Exception as e:
        print(f"\n❌ Group box '{group_name}' encountered a problem: {e}")
        summary[group_name] = {"status": "failed", "error": str(e)}

print("\n" + "="*65)
print("🎉 Master training run finished!")
print("="*65)


Loaded folder: positive_regulation_of_DNA-templated_transcription_GO_0045893
  Proteins counted : 4,796
  Slices: Train=3,836 | Val=480 | Held-out Test=480
  ✓ Already completed. Skipping: positive_regulation_of_DNA-templated_transcription_GO_0045893 (F1: 0.5215)

Loaded folder: regulation_of_mRNA_stability_GO_0043488
  Proteins counted : 466
  Slices: Train=372 | Val=47 | Held-out Test=47
  ✓ Already completed. Skipping: regulation_of_mRNA_stability_GO_0043488 (F1: 0.6647)

Loaded folder: heterochromatin_formation_GO_0031507
  Proteins counted : 609
  Slices: Train=487 | Val=61 | Held-out Test=61
  ✓ Already completed. Skipping: heterochromatin_formation_GO_0031507 (F1: 0.6676)

Loaded folder: pattern_recognition_receptor_signaling_pathway_GO_0002221
  Proteins counted : 306
  Slices: Train=244 | Val=31 | Held-out Test=31
  ✓ Already completed. Skipping: pattern_recognition_receptor_signaling_pathway_GO_0002221 (F1: 0.5589)

Loaded folder: regulation_of_protein_kinase_activity_GO_004

In [21]:
"""
================================================================================
NeuralProt — Training Report Card & Error Diagnostic Tool
================================================================================
What this code does:
1. It reads your 'summary' dictionary to see how every group box performed.
2. It prints a clean table of successful groups and their best F1 scores.
3. It creates a dedicated 'Failed List' showing exactly why those 10-20 groups broke.
4. It flags the big groups that are performing poorly so we can spot them easily.
"""

# Separate lists to hold our findings
successes = []
failures = []

# Walk through the summary box one item at a time
for group_name, details in summary.items():
    if details["status"] == "completed":
        successes.append({
            "name": group_name,
            "f1": details["best_val_f1"],
            "proteins": details["n_proteins"],
            "terms": details["n_go_terms"]
        })
    else:
        failures.append({
            "name": group_name,
            "error": details["error"]
        })

# ── 1. DISPLAY THE SUCCESSFUL GROUPS ──────────────────────────────────────────
print("=" * 80)
print("🏆 NeuralProt Training Success Report Card")
print("=" * 80)
print(f"  {'Group Name':<42} | {'Proteins':<8} | {'GO Terms':<8} | {'Best F1':<8}")
print("-" * 80)

# Sort by F1 score so the best models are at the top
for g in sorted(successes, key=lambda x: x["f1"], reverse=True):
    # Shorten long names so the table stays perfectly aligned
    short_name = g["name"] if len(g["name"]) <= 40 else g["name"][:37] + "..."
    
    # Flag poor performers (F1 below 0.35) with a little warning star
    flag = " ⚠️ Low Score" if g["f1"] < 0.35 else ""
    print(f"  {short_name:<42} | {g['proteins']:<8} | {g['terms']:<8} | {g['f1']:<8.4f}{flag}")

# ── 2. DISPLAY THE BROKEN GROUPS ──────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"🚨 Diagnostic Log — {len(failures)} Broken Groups Found")
print("=" * 80)

if failures:
    for idx, f in enumerate(failures, 1):
        print(f"  {idx}. Group Box : {f['name']}")
        print(f"     Why it broke : {f['error']}")
        print("-" * 80)
else:
    print("  ✅ Zero errors! Every single group folder trained perfectly.")
    print("=" * 80)

🏆 NeuralProt Training Success Report Card
  Group Name                                 | Proteins | GO Terms | Best F1 
--------------------------------------------------------------------------------
  regulation_of_heart_contraction_GO_00...   | 497      | 17       | 0.9047  
  positive_regulation_of_immune_respons...   | 679      | 16       | 0.8930  
  Fallback_molecular_function_6              | 735      | 5        | 0.8917  
  sodium_ion_transmembrane_transporter_...   | 505      | 23       | 0.8906  
  transferase_activity,_transferring_ph...   | 231      | 16       | 0.8783  
  regulation_of_myeloid_cell_differenti...   | 506      | 20       | 0.8720  
  oxidoreductase_activity,_acting_on_NA...   | 480      | 17       | 0.8695  
  carbon-carbon_lyase_activity_GO_0016830    | 302      | 15       | 0.8668  
  hormone_metabolic_process_GO_0042445       | 468      | 16       | 0.8617  
  cation_channel_complex_GO_0034703          | 931      | 16       | 0.8532  
  cell_projection_m